# 03 · Feature engineering

Turns raw landmarks into biomechanical features, then into 2-second windows.

Never used as features: participant id, filename, exercise name, quality label.
Kept as metadata: `video_id`, `subject`, `exercise`, `view`, `quality`.

## Shared setup (run first)

This notebook uses the modules in `ml/`. Upload them once to the Colab runtime:

1. Click the folder icon in the left sidebar (Files).
2. Click **Upload to session storage** and select `extract_pose.py`,
   `fitness_features.py`, `build_dataset.py`, `train_baseline.py`,
   `train_temporal.py`, `export_onnx.py` from `F:\Fitness Coach\ml`.
3. Run the cell below.

The dataset zip is expected at
`/content/drive/MyDrive/Dataset Exercise Quality-wise.zip` (Google Drive).

In [ ]:
import os, sys, subprocess, json, glob
from pathlib import Path

ROOT = Path('/content/fitness')
ROOT.mkdir(parents=True, exist_ok=True)

# The uploaded modules land in /content; mirror them into the work root.
for f in ['extract_pose.py', 'fitness_features.py', 'build_dataset.py', 'train_baseline.py',
          'experiment_generalization.py', 'experiment_clip.py', 'experiment_repmetry.py',
          'train_final.py', 'train_temporal.py', 'export_onnx.py']:
    src = Path('/content') / f
    if src.exists():
        dst = ROOT / f
        if not dst.exists() or src.stat().st_mtime > dst.stat().st_mtime:
            dst.write_bytes(src.read_bytes())

sys.path.insert(0, str(ROOT))
print('work root :', ROOT)
print('modules   :', sorted(p.name for p in ROOT.glob('*.py')))
print('cuda      :', subprocess.run('nvidia-smi -L', shell=True, capture_output=True, text=True).stdout.strip() or 'no GPU')

## 1. Inspect the feature definitions

In [ ]:
import fitness_features as ff
print('per-frame features:', len(ff.FRAME_FEATURE_COLS))
for c in ff.FRAME_FEATURE_COLS:
    print('  ', c)
print()
print('window aggregation stats:', ff.STATS)
print('window feature count   :', len(ff.window_feature_names()))

## 2. Sanity-check the angles on one clip

A squat must show the knee angle swinging from near-straight (~170 deg) to a
deep bend, while the elbow angle stays roughly constant. If this plot is wrong,
every model downstream is wrong - so check it first.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, glob
path = sorted(glob.glob(str(ROOT / 'landmarks/squat/good/side/*.parquet')))[0]
raw = pd.read_parquet(path)
feats = ff.compute_frame_features(raw[raw.detected == 1])
print(path)
feats[['t_sec', 'knee_angle_mean', 'hip_angle_mean', 'torso_inclination', 'knee_bend_abs_mean']].describe().round(1)

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
ax[0].plot(feats.t_sec, feats.knee_angle_l, label='left')
ax[0].plot(feats.t_sec, feats.knee_angle_r, label='right')
ax[0].set_ylabel('knee angle (deg)'); ax[0].legend(); ax[0].set_title('Squat joint angles')
ax[1].plot(feats.t_sec, feats.hip_angle_mean); ax[1].set_ylabel('hip angle (deg)')
ax[2].plot(feats.t_sec, feats.torso_inclination); ax[2].set_ylabel('torso lean (deg)')
ax[2].set_xlabel('seconds'); plt.tight_layout(); plt.show()

## 3. Build the windowed dataset

`build_dataset.py` produces:

* `dataset/windows.parquet` - one row per 2 s window (stride 0.5 s), the unit
  the Random Forest trains on
* `dataset/frames.parquet` - per-frame features, the unit the LSTM trains on
* `dataset/feature_names.json` - the exact and only feature order
* `dataset/per_video.csv` - per-clip pose detection rate

In [ ]:
!cd /content/fitness && python -u /content/fitness/build_dataset.py

## 4. Inspect the resulting tables

In [ ]:
import json, pandas as pd
summary = json.load(open(ROOT / 'dataset/summary.json'))
print(json.dumps(summary, indent=2))
df = pd.read_parquet(ROOT / 'dataset/windows.parquet')
print(df.shape)
print(df[['video_id','exercise','quality','view','subject','label','win_start']].head())
print()
print('label balance per exercise:')
print(df.groupby(['exercise','label']).size().unstack(fill_value=0))
print()
print('windows per video:', df.groupby('video_id').size().describe().round(1).to_dict())

## 5. Leakage audit (must all be True / empty)

In [ ]:
feat_cols = json.load(open(ROOT / 'dataset/feature_names.json'))
forbidden = {'subject', 'video_id', 'filename', 'quality', 'label', 'path'}
print('forbidden names inside feature list:', [c for c in feat_cols if c in forbidden])
print('duplicate feature names:', len(feat_cols) != len(set(feat_cols)))
print('n features:', len(feat_cols))
print('any all-null feature:', [c for c in feat_cols if df[c].isna().all()])